In [7]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# 1. Chargement du dataset AVEC Nutri-Score (notre base d'apprentissage)
# Remarque : j'ai mis sep='\t' (tabulation) au cas où il vienne du même export OFF, 
# mais si c'est un CSV classique que tu as créé, remplace par sep=','
df_train = pd.read_csv('produits_avec_nutriscore.csv', sep='\t', dtype={'code': str}, low_memory=False)

# 2. Chargement du dataset SANS Nutri-Score (celui qu'on veut compléter)
df_predict = pd.read_csv('produits_sans_nutriscore.csv', sep='\t', dtype={'code': str}, low_memory=False)

# Affichage des dimensions pour vérifier que tout est bien chargé
print(f"Taille du dataset d'entraînement (AVEC Nutri-Score) : {df_train.shape}")
print(f"Taille du dataset à compléter (SANS Nutri-Score) : {df_predict.shape}")

Taille du dataset d'entraînement (AVEC Nutri-Score) : (106067, 1)
Taille du dataset à compléter (SANS Nutri-Score) : (33216, 1)


In [10]:
import io

# Liste des colonnes à garder
colonnes_utiles = [
    'code', 'product_name', 'nova_group', 'additives_n',
    'energy-kj_100g', 'energy-kcal_100g', 'sugars_100g', 'carbohydrates_100g',
    'saturated-fat_100g', 'fat_100g', 'salt_100g', 'proteins_100g', 'fiber_100g', 
    'fruits-vegetables-legumes_100g', 'nutriscore_grade' # On garde le grade pour le train
]

colonnes_nutriments = ['sugars_100g', 'carbohydrates_100g', 'saturated-fat_100g', 'fat_100g', 'salt_100g', 'proteins_100g', 'fiber_100g']

def corriger_dataframe_mal_sep(df_input):
    # Cas fréquent: CSV lu avec sep='\t' alors qu'il est séparé par des virgules
    if df_input.shape[1] == 1 and ',' in df_input.columns[0]:
        texte_csv = df_input.columns[0] + '\n' + '\n'.join(df_input.iloc[:, 0].astype(str).tolist())
        return pd.read_csv(io.StringIO(texte_csv), dtype={'code': str}, low_memory=False)
    return df_input

def preparer_donnees(df_input):
    # 0. Correction du parsing si nécessaire
    df_input = corriger_dataframe_mal_sep(df_input)

    # 1. Copie et filtrage des colonnes (en ignorant celles qui n'existent pas dans df_predict)
    cols_presentes = [c for c in colonnes_utiles if c in df_input.columns]
    df = df_input[cols_presentes].copy()
    
    # 2. Conversion des colonnes numériques utiles
    for col in ['nova_group', 'additives_n'] + colonnes_nutriments:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 3. Suppression des lignes sans nom de produit
    df = df.dropna(subset=['product_name'])
    
    # 4. Plafonner à 100g et empêcher les valeurs négatives
    for col in colonnes_nutriments:
        if col in df.columns:
            df.loc[df[col] > 100, col] = 100
            df.loc[df[col] < 0, col] = 0
            
    # 5. Cohérence Sucres/Glucides
    if 'sugars_100g' in df.columns and 'carbohydrates_100g' in df.columns:
        masque_sucre = df['sugars_100g'] > df['carbohydrates_100g']
        df.loc[masque_sucre, 'sugars_100g'] = df.loc[masque_sucre, 'carbohydrates_100g']
        
    # 6. Supprimer les aberrations (Somme > 100g)
    cols_somme = ['carbohydrates_100g', 'fat_100g', 'proteins_100g', 'salt_100g']
    cols_somme_presentes = [c for c in cols_somme if c in df.columns]
    if cols_somme_presentes:
        somme_macros = df[cols_somme_presentes].fillna(0).sum(axis=1)
        df = df[somme_macros <= 100]
    
    return df

# Application de la fonction à nos deux datasets
df_train_clean = preparer_donnees(df_train)
df_predict_clean = preparer_donnees(df_predict)

# Pour l'entraînement, on s'assure de ne garder que des lignes où les nutriments cibles sont connus
colonnes_cibles_presentes = [c for c in colonnes_nutriments if c in df_train_clean.columns]
df_train_clean = df_train_clean.dropna(subset=colonnes_cibles_presentes)

print(f"Train propre : {df_train_clean.shape}")
print(f"Predict propre : {df_predict_clean.shape}")

Train propre : (104313, 15)
Predict propre : (32394, 15)


In [ ]:
# Export des datasets traités en CSV
df_train_clean.to_csv('traité_avec_nutriscore.csv', index=False, encoding='utf-8-sig')
df_predict_clean.to_csv('traité_sans_nutriscore.csv', index=False, encoding='utf-8-sig')

print(f"✅ 'traité_avec_nutriscore.csv' sauvegardé ({len(df_train_clean)} lignes)")
print(f"✅ 'traité_sans_nutriscore.csv' sauvegardé ({len(df_predict_clean)} lignes)")

In [ ]:
# Mots vides francais (stop words) pour TF-IDF
stop_words_fr = [
    'le', 'la', 'les', 'de', 'du', 'des', 'un', 'une', 'et', 'en', 'au', 'aux',
    'a', 'est', 'ce', 'qui', 'que', 'par', 'sur', 'pour', 'avec', 'dans',
    'il', 'elle', 'ils', 'elles', 'on', 'se', 'sa', 'son', 'ses', 'mon', 'ma',
    'mes', 'ton', 'ta', 'tes', 'leur', 'leurs', 'nous', 'vous', 'y', 'si',
    'ne', 'pas', 'plus', 'tres', 'bien', 'ou', 'mais', 'car', 'ni', 'dont',
    'je', 'tu', 'me', 'te', 'lui', 'l', 'd', 'j', 'n', 'qu',
    'sans', 'sous', 'entre', 'vers', 'chez', 'avant', 'apres', 'comme'
]

# Initialisation du vectoriseur TF-IDF (300 mots les plus fréquents)
tfidf = TfidfVectorizer(max_features=300, stop_words=stop_words_fr, lowercase=True)

# 1. On APPREND le vocabulaire sur le df_train
tfidf.fit(df_train_clean['product_name'].fillna(''))

# 2. On TRANSFORME les deux datasets avec ce même vocabulaire
train_matrix = tfidf.transform(df_train_clean['product_name'].fillna(''))
predict_matrix = tfidf.transform(df_predict_clean['product_name'].fillna(''))

# 3. Conversion en DataFrames
train_tfidf_df = pd.DataFrame(
    train_matrix.toarray(), 
    columns=[f"nlp_{word}" for word in tfidf.get_feature_names_out()],
    index=df_train_clean.index
)

predict_tfidf_df = pd.DataFrame(
    predict_matrix.toarray(), 
    columns=[f"nlp_{word}" for word in tfidf.get_feature_names_out()],
    index=df_predict_clean.index
)

# 4. Fusion avec les datasets principaux
df_train_final = pd.concat([df_train_clean, train_tfidf_df], axis=1)
df_predict_final = pd.concat([df_predict_clean, predict_tfidf_df], axis=1)

print("Exemple de mots-clés :", tfidf.get_feature_names_out()[:10])

In [ ]:
# Prenons l'exemple de la prédiction des sucres en premier
cible = 'sugars_100g'

# Variables explicatives (Features) : mots clés NLP + Nova + Additifs
features_cols = [c for c in df_train_final.columns if c.startswith('nlp_')] + ['nova_group', 'additives_n']

# 1. Définition de X et Y sur notre dataset d'entraînement
X = df_train_final[features_cols].fillna(-1) # On remplit les NaN de Nova/Additifs par -1
y = df_train_final[cible]

# 2. Définition du X final sur le dataset à prédire (celui qu'on complétera à la fin)
X_predict_final = df_predict_final[features_cols].fillna(-1)

# 3. Séparation de notre dataset d'entraînement pour la validation (80% Apprentissage / 20% Vérification)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Prêt pour l'entraînement ! \nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_predict_final (les trous à boucher plus tard): {X_predict_final.shape}")